In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
import os

labels_path = r"C:\detection\f1final_datasets\dobby\Alligator\Alligator\train\labels"

classes = set()

for file in os.listdir(labels_path):
    with open(os.path.join(labels_path, file)) as f:
        for line in f:
            cls = int(line.split()[0])
            classes.add(cls)

print(sorted(classes))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [3]:
import os
import shutil
import random

DATASET_PATH = r"C:\detection\f1final_datasets\dobby\Alligator\Alligator\train"

IMG_SRC = os.path.join(DATASET_PATH, "images")
LBL_SRC = os.path.join(DATASET_PATH, "labels")

BASE = "/kaggle/working/dataset_split"

paths = {
    "train": (f"{BASE}/train/images", f"{BASE}/train/labels"),
    "val":   (f"{BASE}/val/images",   f"{BASE}/val/labels"),
    "test":  (f"{BASE}/test/images",  f"{BASE}/test/labels"),
}

# Create all folders
for img_p, lbl_p in paths.values():
    os.makedirs(img_p, exist_ok=True)
    os.makedirs(lbl_p, exist_ok=True)

# Load images
images = [f for f in os.listdir(IMG_SRC) if f.lower().endswith((".jpg",".jpeg",".png"))]
random.shuffle(images)

# Split
n = len(images)
train_split = int(0.7 * n)
val_split = int(0.9 * n)

splits = {
    "train": images[:train_split],
    "val": images[train_split:val_split],
    "test": images[val_split:]
}

# Copy
for split, files in splits.items():
    img_dst, lbl_dst = paths[split]

    count = 0
    for img in files:
        label = os.path.splitext(img)[0] + ".txt"

        if os.path.exists(os.path.join(LBL_SRC, label)):
            shutil.copy(os.path.join(IMG_SRC, img), os.path.join(img_dst, img))
            shutil.copy(os.path.join(LBL_SRC, label), os.path.join(lbl_dst, label))
            count += 1

    print(f"{split}: {count}")

train: 2124
val: 607
test: 304


In [4]:
data_yaml = """
train: /kaggle/working/dataset_split/train/images
val: /kaggle/working/dataset_split/val/images
test: /kaggle/working/dataset_split/test/images

nc: 9
names: ['Longitudinal','Transverse','Alligator','Repaired crack','pothole','crack1','crack2','patchy road','rutting']
"""

with open(r"C:\detection\f1final_datasets\dobby\Alligator\Alligator\data.yaml", "w") as f:
    f.write(data_yaml)

print("data.yaml created")

data.yaml created


In [5]:
! pip install -q ultralytics

In [6]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

model.train(
    data=r"C:\detection\f1final_datasets\dobby\Alligator\Alligator\data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    workers=2   # safer for Kaggle
)

c:\Users\Dell\miniconda3\minic\envs\venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


New https://pypi.org/project/ultralytics/8.4.40 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.30  Python-3.12.12 torch-2.5.1 CUDA:0 (NVIDIA RTX A4000, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\detection\f1final_datasets\dobby\Alligator\Alligator\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001A018F09B20>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.

In [7]:
metrics = model.val()

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Ultralytics 8.4.30  Python-3.12.12 torch-2.5.1 CUDA:0 (NVIDIA RTX A4000, 16376MiB)
Model summary (fused): 93 layers, 25,844,971 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1053.8367.3 MB/s, size: 66.4 KB)
val: Scanning C:\kaggle\working\dataset_split\val\labels.cache... 1797 images, 0 backgrounds, 2 corrupt: 100% ━━━━━━━━━━━━ 1797/1797  0.0s
val: C:\kaggle\working\dataset_split\val\images\Japan_001416_jpg.rf.bc8c0951a9e6e2da21e6e06c0e50944a.jpg: ignoring corrupt image/label: Label class 9 exceeds dataset class count 9. Possible class labels are 0-8
val: C:\kaggle\working\dataset_split\val\images\Japan_012950_jpg.rf.fe9bbc1991e03627845da2abcef99b1f.jpg: ignoring corrupt image/label: Label class 9 exceeds dataset class count 9. Possible class labels are 0-8
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 113/113 5.2it/s 21.7s0.2s
                   all       1795       5498      0.859    

In [8]:
from ultralytics import YOLO

# Load best model
model = YOLO(r"C:\detection\f1final_datasets\dobby\runs\detect\train\weights\best.pt")

# Predict on ALL test images
results = model.predict(
    source="/kaggle/working/dataset_split/test/images",
    save=True,          # saves images with boxes
    save_txt=True,      # saves YOLO txt predictions
    save_conf=True,     # saves confidence scores
    conf=0.25,
    batch=16            # faster
)

print("Prediction done")


image 1/1017 c:\kaggle\working\dataset_split\test\images\Japan_000020_jpg.rf.31f6cec46acd9fe2a8cd4e3d2635e4bb.jpg: 640x640 5 Transverses, 1 Alligator, 1 crack2, 8.2ms
image 2/1017 c:\kaggle\working\dataset_split\test\images\Japan_000025_jpg.rf.cb2c705f66f3cd0766a63db4f95dc7da.jpg: 640x640 1 Alligator, 1 patchy road, 8.2ms
image 3/1017 c:\kaggle\working\dataset_split\test\images\Japan_000042_jpg.rf.bb82cd9f2df25f09664f442b4b361453.jpg: 640x640 2 Alligators, 8.2ms
image 4/1017 c:\kaggle\working\dataset_split\test\images\Japan_000053_jpg.rf.f53935bbaf8980d3d8919466e58c5e26.jpg: 640x640 1 Transverse, 1 Alligator, 8.2ms
image 5/1017 c:\kaggle\working\dataset_split\test\images\Japan_000081_jpg.rf.5a41e72773082817ed27a6e6e3fefee1.jpg: 640x640 1 Longitudinal, 2 Transverses, 2 Alligators, 2 potholes, 2 crack2s, 8.2ms
image 6/1017 c:\kaggle\working\dataset_split\test\images\Japan_000085_jpg.rf.e1dd8fcb3685c5e16f8ca186d5907bba.jpg: 640x640 2 Alligators, 2 potholes, 1 patchy road, 8.2ms
image 7/1

In [9]:
import matplotlib.pyplot as plt
import cv2
import os

pred_path = r"C:\detection\f1final_datasets\dobby\runs\detect\predict"

images = os.listdir(pred_path)

for img in images[:5]:
    img_path = os.path.join(pred_path, img)
    image = cv2.imread(img_path)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(img)
    plt.axis("off")
    plt.show()

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>